# 🎓 Full Fine-tuning v2 (Tutorial-based / KcBERT)

**보정된 UnSmile + 수집된 게임 음성채팅 데이터**로 학습합니다.

- **모델**: `beomi/kcbert-base`
- **메트릭**: `abuse_recall` (통일)
- **데이터**: UnSmile 보정 14,690 + 수집 518 = **15,208건**

In [1]:
import os, torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import precision_recall_fscore_support, label_ranking_average_precision_score
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA: True
GPU: NVIDIA L40S


In [2]:
MODEL_NAME = "beomi/kcbert-base"
OUTPUT_DIR = "./output/full_tutorial_kcbert_v2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 통일된 하이퍼파라미터 (Full FT는 LR 낮춤)
EPOCHS, BATCH_SIZE, LEARNING_RATE = 5, 16, 2e-5
MAX_LENGTH = 128

LABEL_NAMES = ["여성/가족", "남성", "성소수자", "인종/국적", "연령", "지역", "종교", "기타 혐오", "악플/욕설", "clean"]
NUM_LABELS = len(LABEL_NAMES)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
unsmile_train = pd.read_csv("../../3_UnSmile_Correction/unsmile_train_corrected.tsv", sep='\t')
valid_df = pd.read_csv("../../3_UnSmile_Correction/unsmile_valid_corrected.tsv", sep='\t')
collected_df = pd.read_csv("../../1_Data_Labeling_STT/keywords_unsmile_format.tsv", sep='\t')

train_df = pd.concat([unsmile_train, collected_df], ignore_index=True)
print(f"✅ Train: {len(train_df)}건 (보정 {len(unsmile_train)} + 수집 {len(collected_df)})")
print(f"Valid: {len(valid_df)}건")

✅ Train: 15208건 (보정 14690 + 수집 518)
Valid: 3663건


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    tokenized = tokenizer(examples['문장'], padding='max_length', truncation=True, max_length=MAX_LENGTH)
    tokenized['labels'] = [[float(examples[col][i]) for col in LABEL_NAMES] for i in range(len(examples['문장']))]
    return tokenized

train_dataset = Dataset.from_pandas(train_df).map(preprocess_function, batched=True, remove_columns=train_df.columns.tolist())
valid_dataset = Dataset.from_pandas(valid_df).map(preprocess_function, batched=True, remove_columns=valid_df.columns.tolist())

Map:   0%|          | 0/15208 [00:00<?, ? examples/s]

Map:   0%|          | 0/3663 [00:00<?, ? examples/s]

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS, problem_type="multi_label_classification").to(DEVICE)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,} (100% trainable)")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total params: 108,926,218 (100% trainable)


In [6]:
# 통일된 compute_metrics (abuse_recall 포함)
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(predictions)).numpy()
    preds = (probs > 0.5).astype(int)
    labels_int = labels.astype(int)
    _, abuse_r, abuse_f1, _ = precision_recall_fscore_support(labels_int[:,8], preds[:,8], average='binary', zero_division=0)
    _, clean_r, clean_f1, _ = precision_recall_fscore_support(labels_int[:,9], preds[:,9], average='binary', zero_division=0)
    return {'lrap': label_ranking_average_precision_score(labels, predictions), 'abuse_recall': abuse_r, 'abuse_f1': abuse_f1, 'clean_recall': clean_r, 'clean_f1': clean_f1}

In [7]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="abuse_recall", greater_is_better=True,
    logging_steps=50, save_total_limit=2, report_to="none", fp16=True, gradient_accumulation_steps=2
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=valid_dataset,
                  tokenizer=tokenizer, compute_metrics=compute_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

In [8]:
print("🚀 v2 Full FT 학습 시작...")
trainer.train()
print("학습 완료!")

🚀 v2 Full FT 학습 시작...


Epoch,Training Loss,Validation Loss,Lrap,Abuse Recall,Abuse F1,Clean Recall,Clean F1
1,0.246900,0.173756,0.843814,0.550837,0.619392,0.641935,0.701116
2,0.151400,0.138238,0.871769,0.545689,0.643399,0.768817,0.758218
3,0.114700,0.132372,0.871621,0.667954,0.681550,0.682796,0.736659
4,0.098100,0.131352,0.874382,0.670528,0.683279,0.701075,0.741330
5,0.084600,0.130877,0.877177,0.678250,0.687093,0.717204,0.744004


학습 완료!


In [9]:
model.save_pretrained(f"{OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")
print("✅ v2 모델 저장 완료!")

eval_results = trainer.evaluate()
for k, v in eval_results.items(): print(f"  {k}: {v:.4f}")

✅ v2 모델 저장 완료!


  eval_loss: 0.1309
  eval_lrap: 0.8772
  eval_abuse_recall: 0.6782
  eval_abuse_f1: 0.6871
  eval_clean_recall: 0.7172
  eval_clean_f1: 0.7440
  eval_runtime: 3.6566
  eval_samples_per_second: 1001.7640
  eval_steps_per_second: 15.8620
  epoch: 5.0000
